**PDF-Based Intelligent Question Answering System using RAG**

In [1]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
from dotenv import load_dotenv
load_dotenv()


True

In [2]:
print("Google API Key:", bool(os.getenv("GOOGLE_API_KEY")))
print("Groq API Key:", bool(os.getenv("GROQ_API_KEY")))

Google API Key: True
Groq API Key: True


In [4]:
from pypdf import PdfReader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

print("Libraries imported successfully!")

Libraries imported successfully!


Loading and Reading pdf file 

In [16]:
import re
def load_pdf(pdf_path):
    reader = PdfReader(pdf_path)
    documents = []

    for page_number, page in enumerate(reader.pages):
        text = page.extract_text(extraction_mode="layout") or ""

        text = re.sub(r'\s+', ' ', text).strip()

        documents.append(
            Document(
                page_content=text,
                metadata={
                    "source": pdf_path,
                    "page": page_number + 1
                }
            )
        )

    return documents

In [17]:
pdf_path = "sample.pdf"

documents = load_pdf(pdf_path)

print("PDF loaded successfully!")
print("Total pages:", len(documents))

PDF loaded successfully!
Total pages: 9


In [18]:
for doc in documents:
    print(f"\nPage: {doc.metadata['page']}")
    print(doc.page_content)
    
print(documents[0].page_content[:1000])    


Page: 1
CASE STUDY: DISTRIBUTED SYSTEMS, BIG DATA PROCESSING AND HIGH-PERFORMANCE COMPUTING 1. Introduction Distributed systems play an important role in modern computing by allowing multiple computers tocommunicate, share resources, store large amounts of data, and perform complex computations. Instead of depending on a single computer, distributed systems divide tasks among multiple connected machines. Technologies such as Google File System (GFS), Hadoop Distributed File System (HDFS), MapReduce, microservices, and Message Passing Interface (MPI) help organizations handle large-scale data storage, processing, and computational challenges. This case study examines the working principles, importance, advantages, limitations, and real-world applications ofthese technologies in modern distributed computing environments. 2. Distributed File Systems: GFS and HDFS Case Study: Managing Large-Scale Data Storage Modern organizations generate enormous amounts of data through websites, applica

Text Chunking

In [19]:
def split_documents(documents):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200,
        separators=["\n\n", "\n", " ", ""]
    )

    chunks = text_splitter.split_documents(documents)

    return chunks

chunks = split_documents(documents)

print("Text splitting completed!")
print("Total chunks:", len(chunks))

Text splitting completed!
Total chunks: 27


Embedding using GoogleAPIKey 

In [24]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key=os.getenv("GOOGLE_API_KEY")
)


def generate_embeddings(chunks):
    texts = [chunk.page_content for chunk in chunks]

    vectors = embeddings.embed_documents(texts)

    return vectors

In [27]:
from dotenv import load_dotenv
import os

load_dotenv(override=True)

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

In [28]:
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key=GOOGLE_API_KEY
)

In [29]:
all_embeddings = generate_embeddings(chunks)

print("Embeddings generated successfully!")
print("Total embeddings:", len(all_embeddings))
print("Vector dimensions:", len(all_embeddings[0]))

Embeddings generated successfully!
Total embeddings: 27
Vector dimensions: 3072


Store vectors into FAISS Vectordatabase

In [30]:
from langchain_community.vectorstores import FAISS

C:\Users\rohit\AppData\Local\Temp\ipykernel_9204\1059762981.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [32]:
def create_vector_db(chunks, embeddings):
    vector_db = FAISS.from_documents(
        documents=chunks,
        embedding=embeddings
    )

    return vector_db

vector_db = create_vector_db(chunks, embeddings)

print("FAISS vector database created successfully!")
print("Total vectors:", vector_db.index.ntotal)

FAISS vector database created successfully!
Total vectors: 27


Test similarity search

TOP_K 3 Chunks

In [34]:
def search_documents(query, vector_db, k=3):
    results = vector_db.similarity_search(query, k=k)

    return results


results = search_documents(
    "What are distributed systems?",
    vector_db
)

for result in results:
    print(result.page_content)
    print(result.metadata)
    print("-" * 50)

CASE STUDY: DISTRIBUTED SYSTEMS, BIG DATA PROCESSING AND HIGH-PERFORMANCE COMPUTING 1. Introduction Distributed systems play an important role in modern computing by allowing multiple computers tocommunicate, share resources, store large amounts of data, and perform complex computations. Instead of depending on a single computer, distributed systems divide tasks among multiple connected machines. Technologies such as Google File System (GFS), Hadoop Distributed File System (HDFS), MapReduce, microservices, and Message Passing Interface (MPI) help organizations handle large-scale data storage, processing, and computational challenges. This case study examines the working principles, importance, advantages, limitations, and real-world applications ofthese technologies in modern distributed computing environments. 2. Distributed File Systems: GFS and HDFS Case Study: Managing Large-Scale Data Storage Modern organizations generate enormous amounts of data through websites, applications,
{'

Create the retriever function
A retriever fetches the most relevant document chunks from FAISS for a user's question.

In [36]:
def get_retriever(vector_db):
    retriever = vector_db.as_retriever(
        search_type="similarity",
        search_kwargs={"k": 3}
    )
    return retriever


retriever = get_retriever(vector_db)

print("Retriever created successfully!")

Retriever created successfully!


In [37]:
def retrieve_documents(query, retriever):
    results = retriever.invoke(query)
    return results


retrieved_docs = retrieve_documents(
    "What are distributed systems?",
    retriever
)

for doc in retrieved_docs:
    print(doc.page_content)
    print("Metadata:", doc.metadata)
    print("-" * 50)

CASE STUDY: DISTRIBUTED SYSTEMS, BIG DATA PROCESSING AND HIGH-PERFORMANCE COMPUTING 1. Introduction Distributed systems play an important role in modern computing by allowing multiple computers tocommunicate, share resources, store large amounts of data, and perform complex computations. Instead of depending on a single computer, distributed systems divide tasks among multiple connected machines. Technologies such as Google File System (GFS), Hadoop Distributed File System (HDFS), MapReduce, microservices, and Message Passing Interface (MPI) help organizations handle large-scale data storage, processing, and computational challenges. This case study examines the working principles, importance, advantages, limitations, and real-world applications ofthese technologies in modern distributed computing environments. 2. Distributed File Systems: GFS and HDFS Case Study: Managing Large-Scale Data Storage Modern organizations generate enormous amounts of data through websites, applications,
Me

Initialize Groq LLM using API_CALL

In [47]:
from langchain_groq import ChatGroq
import os

def initialize_llm():
    llm = ChatGroq(
        model="openai/gpt-oss-120b",
        groq_api_key=os.getenv("GROQ_API_KEY"),
        temperature=0.3
    )
    return llm


llm = initialize_llm()

print("Groq LLM initialized successfully!")

Groq LLM initialized successfully!


Creating prompt

In [48]:
from langchain_core.prompts import ChatPromptTemplate

def create_prompt():
    prompt = ChatPromptTemplate.from_template("""
    Answer the question using only the provided context.
    If the answer is not available in the context, say
    "I don't know based on the provided document."

    Context:
    {context}

    Question:
    {question}

    Answer:
    """)
    return prompt


prompt = create_prompt()

print("Prompt template created successfully!")

Prompt template created successfully!


In [49]:
def format_documents(docs):
    context = "\n\n".join(
        doc.page_content for doc in docs
    )
    return context


print("Document formatting function created successfully!")

Document formatting function created successfully!


In [50]:
def create_rag_chain(retriever, prompt, llm):
    def rag_pipeline(question):
        docs = retriever.invoke(question)

        context = format_documents(docs)

        messages = prompt.invoke({
            "context": context,
            "question": question
        })

        response = llm.invoke(messages)

        return response.content

    return rag_pipeline


rag_chain = create_rag_chain(retriever, prompt, llm)

print("RAG chain created successfully!")

RAG chain created successfully!


In [60]:
question = " Applications of MPI?"

answer = rag_chain(question)

print("Question:", question)
print("\nAnswer:", answer)

Question:  Applications of MPI?

Answer: MPI is applied in many scientific and engineering domains, including:

- Weather forecasting  
- Climate simulation  
- Computational fluid dynamics (CFD)  
- Nuclear research  
- Molecular modeling  
- Astrophysics  

These fields use MPI to divide large computational problems into smaller tasks that run in parallel and exchange data to produce accurate results.
